# Обучение гибрида v4 (multi-positive)

Обучение гибридной модели (граф + текстовый энкодер) на AI2D с multi-positive контрастным лоссом.

In [ ]:
from pathlib import Path
import os
import sys

def _find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'src' / 'vqa_retrieval').exists() and (candidate / 'experiments').exists():
            return candidate
    raise RuntimeError('Cannot find ai2d_vqa_clean root. Open the notebook from inside the clean project.')

PROJECT_ROOT = _find_project_root()
EXTERNAL_ROOT = PROJECT_ROOT.parent
AI2D_ROOT = EXTERNAL_ROOT / 'ai2d'
DOCVQA_ROOT = EXTERNAL_ROOT / 'docvqa'
INFOGRAPHICVQA_ROOT = EXTERNAL_ROOT / 'infographicvqa'
ROOT = PROJECT_ROOT
os.chdir(PROJECT_ROOT)
src_path = str(PROJECT_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('EXTERNAL_ROOT =', EXTERNAL_ROOT)


# AI2D Hybrid v4 Multipos - Full Code Notebook

This notebook contains the actual training/eval source code used for the hybrid graph model. Source cells are safe to run: script main guards are disabled.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / "runs").exists() and (ROOT.parent / "runs").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT:", ROOT)

RUN_DIR = ROOT / "runs" / "ai2d_hybrid_v4_multipos"
LOG_PATH = ROOT / "runs" / "ai2d_hybrid_v4_multipos_train.log"
METRICS_PATH = RUN_DIR / "metrics.json"
print("RUN_DIR:", RUN_DIR)


## Core Data And Metrics Code

In [ ]:
from __future__ import annotations

import hashlib
import json
import random
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Any, Iterable, Optional, Sequence

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset


def _stable_question_suffix(question: str) -> str:
    digest = hashlib.sha1(question.encode("utf-8")).hexdigest()
    return digest[:12]


def make_question_text(question: str) -> str:
    return f"Question: {str(question).strip()}"


def make_option_text(
    question: str,
    option: str,
    short_description: Optional[str] = None,
    use_caption_context: bool = False,
) -> str:
    text = f"Question: {str(question).strip()} Option: {str(option).strip()}."
    if use_caption_context and short_description:
        ctx = str(short_description).strip()
        if ctx:
            text = f"{text} Context: {ctx}"
    return text


@dataclass(frozen=True)
class Ai2dHybridSample:
    sample_id: str
    image_id: str
    split: str
    image_path: str
    ocr_v2_path: Optional[str]
    short_description: Optional[str]
    question: str
    options: tuple[str, ...]
    correct_option_idx: int
    correct_option_text: str
    text_q_only: str
    text_q_plus_correct: str

    def to_dict(self) -> dict[str, Any]:
        payload = asdict(self)
        payload["options"] = list(self.options)
        return payload

    @staticmethod
    def from_dict(payload: dict[str, Any]) -> "Ai2dHybridSample":
        options = tuple(str(x) for x in payload.get("options", []) if str(x).strip())
        correct_idx = int(payload.get("correct_option_idx", 0))
        if options:
            correct_idx = max(0, min(correct_idx, len(options) - 1))
        else:
            correct_idx = 0
        correct_text = str(payload.get("correct_option_text", "")).strip()
        if options and not correct_text:
            correct_text = options[correct_idx]
        return Ai2dHybridSample(
            sample_id=str(payload.get("sample_id", "")),
            image_id=str(payload.get("image_id", "")),
            split=str(payload.get("split", "all")),
            image_path=str(payload.get("image_path", "")),
            ocr_v2_path=payload.get("ocr_v2_path"),
            short_description=payload.get("short_description"),
            question=str(payload.get("question", "")),
            options=options,
            correct_option_idx=correct_idx,
            correct_option_text=correct_text,
            text_q_only=str(payload.get("text_q_only", "")),
            text_q_plus_correct=str(payload.get("text_q_plus_correct", "")),
        )


def read_test_ids_csv(path: str | Path) -> set[str]:
    path = Path(path)
    out: set[str] = set()
    for line in path.read_text(encoding="utf-8").splitlines():
        token = line.strip()
        if token:
            out.add(token)
    return out


def _load_caption_short_description(caption_path: Path) -> Optional[str]:
    if not caption_path.exists():
        return None
    payload = json.loads(caption_path.read_text(encoding="utf-8"))
    value = str(payload.get("short_description", "")).strip()
    return value or None


def build_hybrid_samples_from_ai2d(
    ai2d_root: str | Path,
    prepared_root: str | Path,
) -> list[Ai2dHybridSample]:
    ai2d_root = Path(ai2d_root)
    prepared_root = Path(prepared_root)
    images_dir = ai2d_root / "images"
    questions_dir = ai2d_root / "questions"
    ocr_dir = prepared_root / "ocr_v2"
    caption_dir = prepared_root / "caption_v1"

    out: list[Ai2dHybridSample] = []
    for q_path in sorted(questions_dir.glob("*.json")):
        payload = json.loads(q_path.read_text(encoding="utf-8"))
        image_name = str(payload.get("imageName", "")).strip()
        if not image_name:
            continue

        image_path = images_dir / image_name
        if not image_path.exists():
            continue

        image_id = Path(image_name).stem
        ocr_path = ocr_dir / f"{image_id}.ocr.json"
        caption_path = caption_dir / f"{image_id}.caption.json"
        short_description = _load_caption_short_description(caption_path)

        questions = payload.get("questions", {})
        for question_text, q_data in questions.items():
            question = str(question_text).strip()
            answers_raw = q_data.get("answerTexts", []) or []
            options = tuple(str(answer).strip() for answer in answers_raw if str(answer).strip())
            if not options:
                continue

            try:
                correct_idx = int(q_data.get("correctAnswer", 0))
            except (TypeError, ValueError):
                correct_idx = 0
            if correct_idx < 0 or correct_idx >= len(options):
                correct_idx = 0

            question_id = str(q_data.get("questionId", "")).strip()
            suffix = question_id or _stable_question_suffix(question)
            sample_id = f"{image_name}:{suffix}"
            correct_text = options[correct_idx]
            q_only = make_question_text(question)
            q_plus_correct = f"{q_only} Correct answer: {correct_text}."

            out.append(
                Ai2dHybridSample(
                    sample_id=sample_id,
                    image_id=image_id,
                    split="all",
                    image_path=str(image_path),
                    ocr_v2_path=str(ocr_path) if ocr_path.exists() else None,
                    short_description=short_description,
                    question=question,
                    options=options,
                    correct_option_idx=correct_idx,
                    correct_option_text=correct_text,
                    text_q_only=q_only,
                    text_q_plus_correct=q_plus_correct,
                )
            )
    return out


def create_image_level_splits(
    image_ids: Iterable[str],
    test_ids: Iterable[str],
    val_ratio: float = 0.1,
    seed: int = 42,
) -> dict[str, Any]:
    unique_image_ids = sorted({str(x).strip() for x in image_ids if str(x).strip()})
    requested_test = {str(x).strip() for x in test_ids if str(x).strip()}
    image_id_set = set(unique_image_ids)

    missing_test_ids = sorted(requested_test - image_id_set)
    test_image_ids = sorted(requested_test & image_id_set)

    remainder = [x for x in unique_image_ids if x not in requested_test]
    rng = random.Random(seed)
    rng.shuffle(remainder)

    if not remainder or val_ratio <= 0:
        val_count = 0
    else:
        val_count = int(len(remainder) * float(val_ratio))
        if len(remainder) > 1:
            val_count = max(1, min(len(remainder) - 1, val_count))
        else:
            val_count = 0

    val_image_ids = sorted(remainder[:val_count])
    train_image_ids = sorted(remainder[val_count:])

    return {
        "seed": int(seed),
        "val_ratio": float(val_ratio),
        "num_images": len(unique_image_ids),
        "train_image_ids": train_image_ids,
        "val_image_ids": val_image_ids,
        "test_image_ids": test_image_ids,
        "missing_test_ids": missing_test_ids,
    }


def assign_splits_to_samples(
    samples: Sequence[Ai2dHybridSample],
    split_payload: dict[str, Any],
) -> list[Ai2dHybridSample]:
    train_ids = set(split_payload.get("train_image_ids", []))
    val_ids = set(split_payload.get("val_image_ids", []))
    test_ids = set(split_payload.get("test_image_ids", []))

    out: list[Ai2dHybridSample] = []
    for sample in samples:
        if sample.image_id in test_ids:
            split = "test"
        elif sample.image_id in val_ids:
            split = "val"
        elif sample.image_id in train_ids:
            split = "train"
        else:
            split = "train"
        out.append(replace(sample, split=split))
    return out


def load_manifest_hybrid(path: str | Path) -> list[Ai2dHybridSample]:
    path = Path(path)
    out: list[Ai2dHybridSample] = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        payload = json.loads(line)
        out.append(Ai2dHybridSample.from_dict(payload))
    return out


def write_manifest_hybrid(samples: Sequence[Ai2dHybridSample], output_path: str | Path) -> Path:
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as handle:
        for sample in samples:
            handle.write(json.dumps(sample.to_dict(), ensure_ascii=False) + "\n")
    return output_path


def load_split_payload(path: str | Path) -> dict[str, Any]:
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_split_payload(payload: dict[str, Any], output_path: str | Path) -> Path:
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    return output_path


def select_samples_for_split(
    samples: Sequence[Ai2dHybridSample],
    split_name: str,
    split_payload: Optional[dict[str, Any]] = None,
) -> list[Ai2dHybridSample]:
    split_name = str(split_name).strip().lower()
    if split_payload is None:
        return [sample for sample in samples if sample.split.lower() == split_name]

    image_ids = set(split_payload.get(f"{split_name}_image_ids", []))
    return [sample for sample in samples if sample.image_id in image_ids]


class Ai2dHybridDataset(Dataset):
    def __init__(self, samples: Sequence[Ai2dHybridSample]) -> None:
        self._samples = list(samples)

    def __len__(self) -> int:
        return len(self._samples)

    def __getitem__(self, idx: int) -> Ai2dHybridSample:
        return self._samples[idx]


def make_hybrid_collate_fn(use_caption_context: bool = False):
    def _collate(batch: Sequence[Ai2dHybridSample]) -> dict[str, Any]:
        if not batch:
            raise ValueError("batch must be non-empty")

        batch_size = len(batch)
        max_options = max(len(sample.options) for sample in batch)
        option_mask = torch.zeros((batch_size, max_options), dtype=torch.bool)
        correct_indices = torch.zeros((batch_size,), dtype=torch.long)
        option_texts: list[list[str]] = []

        for row_idx, sample in enumerate(batch):
            row_texts: list[str] = []
            for col_idx, option in enumerate(sample.options):
                option_mask[row_idx, col_idx] = True
                row_texts.append(
                    make_option_text(
                        question=sample.question,
                        option=option,
                        short_description=sample.short_description,
                        use_caption_context=use_caption_context,
                    )
                )
            while len(row_texts) < max_options:
                row_texts.append("")
            option_texts.append(row_texts)
            correct_indices[row_idx] = int(sample.correct_option_idx)

        return {
            "sample_ids": [sample.sample_id for sample in batch],
            "image_ids": [sample.image_id for sample in batch],
            "splits": [sample.split for sample in batch],
            "image_paths": [sample.image_path for sample in batch],
            "ocr_paths": [sample.ocr_v2_path for sample in batch],
            "questions": [sample.question for sample in batch],
            "question_texts": [sample.text_q_only for sample in batch],
            "short_descriptions": [sample.short_description for sample in batch],
            "options": [list(sample.options) for sample in batch],
            "option_texts": option_texts,
            "option_mask": option_mask,
            "correct_indices": correct_indices,
            "correct_texts": [sample.correct_option_text for sample in batch],
        }

    return _collate


def flatten_option_texts(
    option_texts: Sequence[Sequence[str]],
    option_mask: torch.Tensor,
) -> tuple[list[str], list[tuple[int, int]]]:
    flat_texts: list[str] = []
    flat_positions: list[tuple[int, int]] = []
    for row_idx, row in enumerate(option_texts):
        for col_idx, text in enumerate(row):
            if bool(option_mask[row_idx, col_idx]):
                flat_texts.append(str(text))
                flat_positions.append((row_idx, col_idx))
    return flat_texts, flat_positions


def encode_text_batch(
    texts: Sequence[str],
    text_encoder,
    cache=None,
    normalize: bool = False,
) -> torch.Tensor:
    if cache is not None and hasattr(cache, "get_text_batch"):
        return cache.get_text_batch(list(texts), text_encoder, normalize=normalize)
    with torch.no_grad():
        embs = text_encoder.encode(list(texts), convert_to_tensor=True, normalize_embeddings=normalize)
    return embs.detach().cpu() if isinstance(embs, torch.Tensor) else torch.tensor(embs, dtype=torch.float32)


def compute_option_logits(
    z_img: torch.Tensor,
    option_texts: Sequence[Sequence[str]],
    option_mask: torch.Tensor,
    text_encoder,
    text_proj,
    cache=None,
    temperature: float = 0.07,
) -> torch.Tensor:
    if z_img.dim() != 2:
        raise ValueError(f"z_img must be 2D [B, D], got shape {tuple(z_img.shape)}")

    batch_size, _ = z_img.shape
    max_options = option_mask.shape[1]
    logits = torch.full(
        (batch_size, max_options),
        fill_value=-1e9,
        dtype=z_img.dtype,
        device=z_img.device,
    )

    flat_texts, flat_positions = flatten_option_texts(option_texts, option_mask)
    if not flat_texts:
        return logits

    text_emb = encode_text_batch(flat_texts, text_encoder=text_encoder, cache=cache, normalize=False).to(z_img.device)
    z_opt = F.normalize(text_proj(text_emb), dim=1)

    for emb_idx, (row_idx, col_idx) in enumerate(flat_positions):
        logits[row_idx, col_idx] = torch.dot(z_img[row_idx], z_opt[emb_idx]) / temperature
    return logits


def positive_mask_from_group_ids(group_ids: Sequence[str], device: Optional[torch.device] = None) -> torch.Tensor:
    ids = [str(x) for x in group_ids]
    if not ids:
        raise ValueError("group_ids must be non-empty")
    return torch.tensor(
        [[left == right for right in ids] for left in ids],
        dtype=torch.bool,
        device=device,
    )


def _multi_positive_ce(logits: torch.Tensor, positive_mask: torch.Tensor) -> torch.Tensor:
    if logits.shape != positive_mask.shape:
        raise ValueError(
            f"positive_mask must have shape {tuple(logits.shape)}, got {tuple(positive_mask.shape)}"
        )
    positive_mask = positive_mask.to(device=logits.device, dtype=torch.bool)
    if not bool(positive_mask.any(dim=1).all()):
        raise ValueError("each row must contain at least one positive")

    log_probs = logits - torch.logsumexp(logits, dim=1, keepdim=True)
    positive_log_probs = torch.logsumexp(
        log_probs.masked_fill(~positive_mask, torch.finfo(logits.dtype).min),
        dim=1,
    )
    return -positive_log_probs.mean()


def contrastive_loss(
    z_img: torch.Tensor,
    z_txt: torch.Tensor,
    temperature: float = 0.07,
    positive_mask: Optional[torch.Tensor] = None,
    group_ids: Optional[Sequence[str]] = None,
) -> torch.Tensor:
    logits = (z_img @ z_txt.t()) / temperature
    if positive_mask is None:
        if group_ids is not None:
            positive_mask = positive_mask_from_group_ids(group_ids, device=logits.device)
        else:
            labels = torch.arange(logits.size(0), device=logits.device)
            positive_mask = F.one_hot(labels, num_classes=logits.size(1)).bool()
    else:
        positive_mask = positive_mask.to(logits.device)
    return (
        _multi_positive_ce(logits, positive_mask)
        + _multi_positive_ce(logits.t(), positive_mask.t())
    ) / 2.0


def recall_at_k_torch(
    sim: torch.Tensor,
    ks: Sequence[int] = (1, 5, 10),
    positive_mask: Optional[torch.Tensor] = None,
) -> dict[int, float]:
    if positive_mask is None:
        gt = torch.arange(sim.size(0), device=sim.device).unsqueeze(1)
    else:
        if positive_mask.shape != sim.shape:
            raise ValueError(
                f"positive_mask must have shape {tuple(sim.shape)}, got {tuple(positive_mask.shape)}"
            )
        positive_mask = positive_mask.to(device=sim.device, dtype=torch.bool)
        if not bool(positive_mask.any(dim=1).all()):
            raise ValueError("each row must contain at least one positive")

    ranks = sim.argsort(dim=1, descending=True)
    out: dict[int, float] = {}
    for k in ks:
        k_eff = min(int(k), sim.size(0))
        if positive_mask is None:
            hits = (ranks[:, :k_eff] == gt).any(dim=1)
        else:
            hits = positive_mask.gather(dim=1, index=ranks[:, :k_eff]).any(dim=1)
        out[int(k)] = float(hits.float().mean().item())
    return out


def retrieval_metrics_from_embeddings(
    z_img: torch.Tensor,
    z_txt: torch.Tensor,
    ks: Sequence[int] = (1, 5, 10),
    image_ids: Optional[Sequence[str]] = None,
    positive_mask: Optional[torch.Tensor] = None,
) -> dict[str, Any]:
    sim = z_img @ z_txt.t()
    if positive_mask is None and image_ids is not None:
        positive_mask = positive_mask_from_group_ids(image_ids, device=sim.device)
    if positive_mask is not None:
        positive_mask = positive_mask.to(device=sim.device, dtype=torch.bool)
    i2t = recall_at_k_torch(sim, ks=ks, positive_mask=positive_mask)
    t2i = recall_at_k_torch(
        sim.t(),
        ks=ks,
        positive_mask=positive_mask.t() if positive_mask is not None else None,
    )
    mean = {int(k): 0.5 * (i2t[int(k)] + t2i[int(k)]) for k in ks}
    return {"i2t": i2t, "t2i": t2i, "mean": mean, "sim": sim}


def vqa_accuracy_from_logits(logits: torch.Tensor, targets: torch.Tensor) -> float:
    preds = logits.argmax(dim=1)
    return float((preds == targets).float().mean().item())


## Graph Builder / GAT Encoder Code

In [ ]:
from __future__ import annotations

import hashlib
import inspect
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import cv2
import numpy as np
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sentence_transformers import SentenceTransformer
from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv, GlobalAttention, global_mean_pool
from torchvision import transforms


@dataclass
class NodeV2:
    bbox: Tuple[int, int, int, int]
    kind: str
    text: str = ""
    conf: float = 0.0


def bbox_center_v2(bbox: Tuple[int, int, int, int]) -> Tuple[float, float]:
    x1, y1, x2, y2 = bbox
    return (x1 + x2) / 2.0, (y1 + y2) / 2.0


def detect_shapes_opencv_v2(
    img_bgr: np.ndarray,
    min_area: int = 300,
    max_nodes: int = 80,
) -> List[Tuple[int, int, int, int]]:
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (3, 3), 0)
    thr = cv2.adaptiveThreshold(
        gray,
        255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY_INV,
        31,
        7,
    )
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    thr = cv2.morphologyEx(thr, cv2.MORPH_CLOSE, kernel, iterations=1)

    contours, _ = cv2.findContours(thr, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes: list[Tuple[int, int, int, int]] = []
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        if w * h < min_area:
            continue
        boxes.append((x, y, x + w, y + h))

    boxes.sort(key=lambda bbox: (bbox[2] - bbox[0]) * (bbox[3] - bbox[1]), reverse=True)
    return boxes[:max_nodes]


def parse_ocr_v2_json(
    ocr_path: str | Path,
    level: str = "line",
    min_conf: float = 0.0,
) -> List[NodeV2]:
    ocr_path = Path(ocr_path)
    with ocr_path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)

    if level not in {"line", "word"}:
        raise ValueError(f"Unsupported OCR level: {level}")

    key = "lines" if level == "line" else "words"
    nodes: list[NodeV2] = []
    for item in payload.get(key, []):
        text = str(item.get("text", "")).strip()
        conf = float(item.get("conf", 0.0))
        bbox_raw = item.get("bbox", [])
        if not text or conf < min_conf or len(bbox_raw) != 4:
            continue
        bbox = tuple(int(v) for v in bbox_raw)
        nodes.append(NodeV2(bbox=bbox, kind="text", text=text, conf=conf))
    return nodes


def build_edges_knn_v2(nodes: List[NodeV2], k: int = 4) -> torch.Tensor:
    centers = np.array([bbox_center_v2(node.bbox) for node in nodes], dtype=np.float32)
    if len(centers) == 0:
        return torch.empty((2, 0), dtype=torch.long)

    dists = np.sqrt(((centers[:, None, :] - centers[None, :, :]) ** 2).sum(-1))
    np.fill_diagonal(dists, np.inf)

    edges: list[tuple[int, int]] = []
    for idx in range(len(nodes)):
        for nbr in np.argsort(dists[idx])[:k]:
            edges.append((idx, int(nbr)))
            edges.append((int(nbr), idx))

    if not edges:
        return torch.empty((2, 0), dtype=torch.long)
    return torch.tensor(edges, dtype=torch.long).t().contiguous()


class NodeFeaturizerV2:
    def __init__(
        self,
        device: str = "cpu",
        vision_model_name: str = "vit_base_patch16_224",
        text_model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
    ) -> None:
        self.device = torch.device(device)
        self.vision_model_name = vision_model_name
        self.text_model_name = text_model_name

        self.vision = timm.create_model(vision_model_name, pretrained=True, num_classes=0)
        self.vision.eval().to(self.device)
        self.vision_dim = self.vision.num_features

        self.text_enc = SentenceTransformer(text_model_name, device=str(self.device))
        self.text_dim = self.text_enc.get_sentence_embedding_dimension()

        self.preprocess = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ])

    @torch.no_grad()
    def _visual_emb(self, pil_img: Image.Image) -> torch.Tensor:
        x = self.preprocess(pil_img).unsqueeze(0).to(self.device)
        return self.vision(x).squeeze(0).detach().cpu()

    @torch.no_grad()
    def _text_emb(self, text: str) -> torch.Tensor:
        if not text:
            return torch.zeros(self.text_dim, dtype=torch.float32)
        emb = self.text_enc.encode([text], convert_to_tensor=True, normalize_embeddings=False)
        return emb.squeeze(0).detach().cpu()

    def _geom_feat(self, bbox: Tuple[int, int, int, int], width: int, height: int) -> torch.Tensor:
        x1, y1, x2, y2 = bbox
        cx, cy = bbox_center_v2(bbox)
        w, h = max(1, x2 - x1), max(1, y2 - y1)
        return torch.tensor([
            cx / width,
            cy / height,
            w / width,
            h / height,
            (w * h) / (width * height),
            x1 / width,
            y1 / height,
            x2 / width,
            y2 / height,
            max(0.0, min(1.0, float(w) / max(1.0, float(h)))),
            max(0.0, min(1.0, float(h) / max(1.0, float(w)))),
            1.0 if x1 <= 2 or y1 <= 2 else 0.0,
        ], dtype=torch.float32)

    def extract(self, pil_img: Image.Image, nodes: List[NodeV2]) -> torch.Tensor:
        width, height = pil_img.size
        feats = []
        for node in nodes:
            crop = pil_img.crop(node.bbox)
            visual = self._visual_emb(crop)
            text = self._text_emb(node.text if node.kind == "text" else "")
            geom = self._geom_feat(node.bbox, width, height)
            kind_flag = torch.tensor([1.0 if node.kind == "text" else 0.0], dtype=torch.float32)
            conf_feat = torch.tensor([max(0.0, min(1.0, node.conf / 100.0))], dtype=torch.float32)
            feats.append(torch.cat([visual, text, geom, kind_flag, conf_feat]))
        if not feats:
            return torch.zeros((1, self.vision_dim + self.text_dim + 12 + 1 + 1), dtype=torch.float32)
        return torch.stack(feats)


def build_graph_v2(
    image_path: str | Path,
    featurizer: 'NodeFeaturizerV2',
    ocr_path: Optional[str | Path] = None,
    ocr_level: str = "line",
    min_area: int = 300,
    max_shape_nodes: int = 80,
    min_text_conf: float = 35.0,
    knn_k: int = 4,
    include_shapes: bool = True,
) -> Data:
    image_path = str(Path(image_path))
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise RuntimeError(f"cv2.imread failed: {image_path}")

    pil_img = Image.open(image_path).convert("RGB")

    shape_nodes: list[NodeV2] = []
    if include_shapes:
        shape_nodes = [
            NodeV2(bbox=bbox, kind="shape")
            for bbox in detect_shapes_opencv_v2(img_bgr, min_area=min_area, max_nodes=max_shape_nodes)
        ]

    text_nodes: list[NodeV2] = []
    if ocr_path:
        text_nodes = parse_ocr_v2_json(ocr_path, level=ocr_level, min_conf=min_text_conf)

    nodes = shape_nodes + text_nodes
    edge_index = build_edges_knn_v2(nodes, k=knn_k)
    x = featurizer.extract(pil_img, nodes)
    return Data(x=x, edge_index=edge_index)


class GraphEncoderV2(nn.Module):
    def __init__(
        self,
        in_dim: int,
        hidden_dim: int = 256,
        out_dim: int = 256,
        num_heads: int = 4,
        use_attn_pool: bool = True,
    ) -> None:
        super().__init__()
        self.proj = nn.Linear(in_dim, hidden_dim)
        self.gnn1 = GATv2Conv(hidden_dim, hidden_dim // num_heads, heads=num_heads, dropout=0.1)
        self.gnn2 = GATv2Conv(hidden_dim, hidden_dim // num_heads, heads=num_heads, dropout=0.1)
        self.out = nn.Linear(hidden_dim, out_dim)
        self.use_attn_pool = use_attn_pool
        gate_nn = nn.Sequential(
            nn.Linear(out_dim, max(1, out_dim // 2)),
            nn.GELU(),
            nn.Linear(max(1, out_dim // 2), 1),
        )
        self.pool = GlobalAttention(gate_nn=gate_nn)

    def forward(self, batch) -> torch.Tensor:
        x, edge_index, batch_idx = batch.x, batch.edge_index, batch.batch
        x = F.gelu(self.proj(x))
        x = F.gelu(self.gnn1(x, edge_index))
        x = F.gelu(self.gnn2(x, edge_index))
        x = F.gelu(self.out(x))
        return self.pool(x, batch_idx) if self.use_attn_pool else global_mean_pool(x, batch_idx)


class FeatureCacheV2:
    def __init__(self, cache_dir: str | Path, signature: str, enabled: bool = True) -> None:
        self.enabled = enabled
        self.signature = str(signature)
        self.root = Path(cache_dir)
        self.graph_dir = self.root / self.signature / "graphs"
        self.text_dir = self.root / self.signature / "texts"
        self.graph_dir.mkdir(parents=True, exist_ok=True)
        self.text_dir.mkdir(parents=True, exist_ok=True)
        self._mem_graph: Dict[str, Data] = {}
        self._mem_text: Dict[str, torch.Tensor] = {}

    @staticmethod
    def _sha1(value: str) -> str:
        return hashlib.sha1(value.encode("utf-8")).hexdigest()

    @staticmethod
    def make_signature(
        vision_model_name: str,
        text_model_name: str,
        ocr_level: str,
        min_area: int,
        max_shape_nodes: int,
        min_text_conf: float,
        knn_k: int,
        include_shapes: bool,
    ) -> str:
        payload = {
            "vision_model_name": vision_model_name,
            "text_model_name": text_model_name,
            "ocr_level": ocr_level,
            "min_area": int(min_area),
            "max_shape_nodes": int(max_shape_nodes),
            "min_text_conf": float(min_text_conf),
            "knn_k": int(knn_k),
            "include_shapes": bool(include_shapes),
            "cache_version": 1,
        }
        raw = json.dumps(payload, sort_keys=True, ensure_ascii=False)
        return hashlib.sha1(raw.encode("utf-8")).hexdigest()[:16]

    def _load_pt(self, path: Path):
        if "weights_only" in inspect.signature(torch.load).parameters:
            return torch.load(path, map_location="cpu", weights_only=False)
        return torch.load(path, map_location="cpu")

    def get_graph(
        self,
        image_path: str | Path,
        featurizer: 'NodeFeaturizerV2',
        ocr_path: Optional[str | Path] = None,
        ocr_level: str = "line",
        min_area: int = 300,
        max_shape_nodes: int = 80,
        min_text_conf: float = 35.0,
        knn_k: int = 4,
        include_shapes: bool = True,
    ) -> Data:
        if not self.enabled:
            return build_graph_v2(
                image_path=image_path,
                featurizer=featurizer,
                ocr_path=ocr_path,
                ocr_level=ocr_level,
                min_area=min_area,
                max_shape_nodes=max_shape_nodes,
                min_text_conf=min_text_conf,
                knn_k=knn_k,
                include_shapes=include_shapes,
            )

        image_path = Path(image_path)
        ocr_file = Path(ocr_path) if ocr_path else None

        try:
            image_mtime = image_path.stat().st_mtime_ns
            image_resolved = str(image_path.resolve())
        except FileNotFoundError:
            image_mtime = 0
            image_resolved = str(image_path)

        if ocr_file is not None:
            try:
                ocr_mtime = ocr_file.stat().st_mtime_ns
                ocr_resolved = str(ocr_file.resolve())
            except FileNotFoundError:
                ocr_mtime = 0
                ocr_resolved = str(ocr_file)
        else:
            ocr_mtime = 0
            ocr_resolved = ""

        key = self._sha1(
            "|".join([
                image_resolved,
                str(image_mtime),
                ocr_resolved,
                str(ocr_mtime),
                ocr_level,
                str(min_area),
                str(max_shape_nodes),
                str(min_text_conf),
                str(knn_k),
                str(include_shapes),
                self.signature,
            ])
        )

        if key in self._mem_graph:
            return self._mem_graph[key]

        fpath = self.graph_dir / f"{key}.pt"
        if fpath.exists():
            graph = self._load_pt(fpath)
        else:
            graph = build_graph_v2(
                image_path=image_path,
                featurizer=featurizer,
                ocr_path=ocr_path,
                ocr_level=ocr_level,
                min_area=min_area,
                max_shape_nodes=max_shape_nodes,
                min_text_conf=min_text_conf,
                knn_k=knn_k,
                include_shapes=include_shapes,
            )
            torch.save(graph, fpath)

        self._mem_graph[key] = graph
        return graph

    def get_text_batch(
        self,
        texts: List[str],
        text_encoder,
        normalize: bool = False,
    ) -> torch.Tensor:
        if not self.enabled:
            with torch.no_grad():
                embs = text_encoder.encode(texts, convert_to_tensor=True, normalize_embeddings=normalize)
            return embs.detach().cpu() if isinstance(embs, torch.Tensor) else torch.tensor(embs)

        order_keys: list[str] = []
        missing_texts: list[str] = []
        missing_keys: list[str] = []

        for text in texts:
            text = text or ""
            key = self._sha1(f"{text}|{self.signature}")
            order_keys.append(key)
            if key in self._mem_text:
                continue
            fpath = self.text_dir / f"{key}.pt"
            if fpath.exists():
                self._mem_text[key] = self._load_pt(fpath)
            else:
                missing_texts.append(text)
                missing_keys.append(key)

        if missing_texts:
            with torch.no_grad():
                embs = text_encoder.encode(missing_texts, convert_to_tensor=True, normalize_embeddings=normalize)
            embs = embs.detach().cpu() if isinstance(embs, torch.Tensor) else torch.tensor(embs)
            for key, emb in zip(missing_keys, embs):
                emb = emb.contiguous()
                self._mem_text[key] = emb
                torch.save(emb, self.text_dir / f"{key}.pt")

        return torch.stack([self._mem_text[key] for key in order_keys])


def resolve_ocr_v2_path(
    image_path: str | Path,
    image_root: str | Path,
    ocr_root: str | Path,
    suffix: str = ".ocr.json",
) -> Path:
    image_path = Path(image_path)
    image_root = Path(image_root)
    ocr_root = Path(ocr_root)
    relative = image_path.relative_to(image_root)
    return ocr_root / relative.with_suffix(suffix)



## Full Training Script Code

In [ ]:
from __future__ import annotations

import argparse
import json
import random
import sys
from contextlib import nullcontext
from pathlib import Path
from typing import Any, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch_geometric.data import Batch
from tqdm.auto import tqdm

ROOT = Path.cwd()
if not (ROOT / "runs").exists() and (ROOT.parent / "runs").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from vqa_retrieval.ai2d_hybrid import (
    Ai2dHybridDataset,
    compute_option_logits,
    contrastive_loss,
    load_manifest_hybrid,
    load_split_payload,
    make_hybrid_collate_fn,
    retrieval_metrics_from_embeddings,
    select_samples_for_split,
    vqa_accuracy_from_logits,
)
from vqa_retrieval.graph_builder_v2 import FeatureCacheV2, GraphEncoderV2, NodeFeaturizerV2


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def maybe_limit_samples(
    samples: list,
    limit: Optional[int],
    seed: int,
) -> list:
    if limit is None or limit <= 0 or len(samples) <= limit:
        return samples
    rng = random.Random(seed)
    idx = list(range(len(samples)))
    rng.shuffle(idx)
    idx = sorted(idx[:limit])
    return [samples[i] for i in idx]


def write_jsonl(rows: list[dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")


def build_cache_signature(args, featurizer: 'NodeFeaturizerV2') -> str:
    return FeatureCacheV2.make_signature(
        vision_model_name=featurizer.vision_model_name,
        text_model_name=featurizer.text_model_name,
        ocr_level=args.ocr_level,
        min_area=args.extract_min_area,
        max_shape_nodes=args.extract_max_nodes,
        min_text_conf=args.extract_min_text_conf,
        knn_k=args.extract_knn_k,
        include_shapes=not args.disable_shape_nodes,
    )


def encode_question_embeddings(batch, featurizer: 'NodeFeaturizerV2', cache: FeatureCacheV2, device: torch.device):
    q_emb = cache.get_text_batch(batch["question_texts"], featurizer.text_enc, normalize=False)
    return q_emb.to(device)


def build_graph_batch(
    batch: dict[str, Any],
    cache: FeatureCacheV2,
    featurizer: 'NodeFeaturizerV2',
    args,
    device: torch.device,
) -> Batch:
    graphs = [
        cache.get_graph(
            image_path=image_path,
            featurizer=featurizer,
            ocr_path=ocr_path,
            ocr_level=args.ocr_level,
            min_area=args.extract_min_area,
            max_shape_nodes=args.extract_max_nodes,
            min_text_conf=args.extract_min_text_conf,
            knn_k=args.extract_knn_k,
            include_shapes=not args.disable_shape_nodes,
        )
        for image_path, ocr_path in zip(batch["image_paths"], batch["ocr_paths"])
    ]
    return Batch.from_data_list(graphs).to(device)


def evaluate_split(
    loader: DataLoader,
    gnn: GraphEncoderV2,
    text_proj: nn.Module,
    featurizer: 'NodeFeaturizerV2',
    cache: FeatureCacheV2,
    args,
    device: torch.device,
    split_name: str,
) -> dict[str, Any]:
    gnn.eval()
    text_proj.eval()
    all_img_emb: list[torch.Tensor] = []
    all_q_emb: list[torch.Tensor] = []
    ordered_sample_ids: list[str] = []
    ordered_image_ids: list[str] = []
    vqa_predictions: list[dict[str, Any]] = []

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"[eval:{split_name}]", leave=False):
            batch_graph = build_graph_batch(batch, cache, featurizer, args, device)
            q_emb = encode_question_embeddings(batch, featurizer, cache, device)

            z_img = F.normalize(gnn(batch_graph), dim=1)
            z_q = F.normalize(text_proj(q_emb), dim=1)

            logits = compute_option_logits(
                z_img=z_img,
                option_texts=batch["option_texts"],
                option_mask=batch["option_mask"],
                text_encoder=featurizer.text_enc,
                text_proj=text_proj,
                cache=cache,
                temperature=args.temperature,
            )
            targets = batch["correct_indices"].to(device)
            preds = logits.argmax(dim=1)

            all_img_emb.append(z_img.detach().cpu())
            all_q_emb.append(z_q.detach().cpu())
            ordered_sample_ids.extend(batch["sample_ids"])
            ordered_image_ids.extend(batch["image_ids"])

            for i, sample_id in enumerate(batch["sample_ids"]):
                options = batch["options"][i]
                pred_idx = int(preds[i].item())
                gold_idx = int(targets[i].item())
                pred_text = options[pred_idx] if 0 <= pred_idx < len(options) else ""
                gold_text = options[gold_idx] if 0 <= gold_idx < len(options) else ""
                vqa_predictions.append(
                    {
                        "sample_id": sample_id,
                        "image_id": batch["image_ids"][i],
                        "question": batch["questions"][i],
                        "pred_option_idx": pred_idx,
                        "pred_option_text": pred_text,
                        "gold_option_idx": gold_idx,
                        "gold_option_text": gold_text,
                        "is_correct": bool(pred_idx == gold_idx),
                    }
                )

    if not all_img_emb:
        raise RuntimeError(f"No batches for split={split_name}.")

    z_img = torch.cat(all_img_emb, dim=0)
    z_q = torch.cat(all_q_emb, dim=0)
    retrieval = retrieval_metrics_from_embeddings(
        z_img=z_img,
        z_txt=z_q,
        ks=(1, 5, 10),
        image_ids=ordered_image_ids,
    )
    sim = retrieval["sim"]

    logits_tensor = torch.tensor([1.0 if x["is_correct"] else 0.0 for x in vqa_predictions], dtype=torch.float32)
    vqa_acc = float(logits_tensor.mean().item()) if len(logits_tensor) else 0.0
    composite = 0.5 * float(retrieval["mean"][10]) + 0.5 * vqa_acc

    top_k = min(10, sim.size(1))
    top_idx = sim.topk(k=top_k, dim=1).indices
    retrieval_predictions: list[dict[str, Any]] = []
    for row_idx in range(sim.size(0)):
        ranks = [ordered_image_ids[int(col_idx)] for col_idx in top_idx[row_idx].tolist()]
        own_image_id = ordered_image_ids[row_idx]
        retrieval_predictions.append(
            {
                "sample_id": ordered_sample_ids[row_idx],
                "image_id": own_image_id,
                "top10_image_ids": ranks,
                "hit@10": bool(own_image_id in ranks),
            }
        )

    metrics = {
        "split": split_name,
        "i2t": {str(k): float(v) for k, v in retrieval["i2t"].items()},
        "t2i": {str(k): float(v) for k, v in retrieval["t2i"].items()},
        "mean": {str(k): float(v) for k, v in retrieval["mean"].items()},
        "vqa_acc": float(vqa_acc),
        "composite": float(composite),
    }
    return {
        "metrics": metrics,
        "vqa_predictions": vqa_predictions,
        "retrieval_predictions": retrieval_predictions,
    }


def train_epoch(
    loader: DataLoader,
    gnn: GraphEncoderV2,
    text_proj: nn.Module,
    featurizer: 'NodeFeaturizerV2',
    cache: FeatureCacheV2,
    optimizer: torch.optim.Optimizer,
    scaler: torch.cuda.amp.GradScaler,
    args,
    device: torch.device,
    stage: str,
) -> dict[str, float]:
    gnn.train()
    text_proj.train()

    total_loss = 0.0
    total_ret = 0.0
    total_vqa = 0.0
    n_steps = 0

    use_amp = scaler.is_enabled()
    autocast_ctx = torch.cuda.amp.autocast if use_amp else nullcontext
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(loader, desc=f"[train:{stage}]", leave=False)
    for step, batch in enumerate(pbar, start=1):
        batch_graph = build_graph_batch(batch, cache, featurizer, args, device)
        q_emb = encode_question_embeddings(batch, featurizer, cache, device)

        with autocast_ctx():
            z_img = F.normalize(gnn(batch_graph), dim=1)
            z_q = F.normalize(text_proj(q_emb), dim=1)
            loss_ret = contrastive_loss(
                z_img,
                z_q,
                temperature=args.temperature,
                group_ids=batch["image_ids"],
            )

            if stage == "stage2":
                logits = compute_option_logits(
                    z_img=z_img,
                    option_texts=batch["option_texts"],
                    option_mask=batch["option_mask"],
                    text_encoder=featurizer.text_enc,
                    text_proj=text_proj,
                    cache=cache,
                    temperature=args.temperature,
                )
                targets = batch["correct_indices"].to(device)
                loss_vqa = F.cross_entropy(logits, targets)
                loss = args.lambda_ret * loss_ret + args.lambda_vqa * loss_vqa
            else:
                loss_vqa = torch.zeros_like(loss_ret)
                loss = loss_ret

        loss_for_backward = loss / max(1, args.grad_accum_steps)
        if scaler.is_enabled():
            scaler.scale(loss_for_backward).backward()
        else:
            loss_for_backward.backward()

        if step % max(1, args.grad_accum_steps) == 0 or step == len(loader):
            if scaler.is_enabled():
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        total_loss += float(loss.detach().item())
        total_ret += float(loss_ret.detach().item())
        total_vqa += float(loss_vqa.detach().item())
        n_steps += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}", ret=f"{loss_ret.item():.4f}", vqa=f"{loss_vqa.item():.4f}")

    denom = max(1, n_steps)
    return {
        "loss": total_loss / denom,
        "loss_ret": total_ret / denom,
        "loss_vqa": total_vqa / denom,
    }


def save_checkpoint(
    path: Path,
    gnn: GraphEncoderV2,
    text_proj: nn.Module,
    optimizer: torch.optim.Optimizer,
    featurizer: 'NodeFeaturizerV2',
    cache_signature: str,
    args,
    epoch_idx: int,
    stage: str,
    val_metrics: dict[str, Any],
) -> None:
    payload = {
        "epoch": epoch_idx,
        "stage": stage,
        "gnn_state_dict": gnn.state_dict(),
        "text_proj_state_dict": text_proj.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "cache_signature": cache_signature,
        "model_config": {
            "in_dim": int(args.in_dim),
            "hidden_dim": int(args.hidden_dim),
            "out_dim": int(args.out_dim),
            "use_attn_pool": bool(not args.disable_attn_pool),
            "vision_model_name": featurizer.vision_model_name,
            "text_model_name": featurizer.text_model_name,
            "ocr_level": args.ocr_level,
            "extract_min_area": int(args.extract_min_area),
            "extract_max_nodes": int(args.extract_max_nodes),
            "extract_min_text_conf": float(args.extract_min_text_conf),
            "extract_knn_k": int(args.extract_knn_k),
            "include_shapes": bool(not args.disable_shape_nodes),
            "temperature": float(args.temperature),
            "use_caption_context": bool(args.use_caption_context),
        },
        "val_metrics": val_metrics,
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(payload, path)


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Train hybrid AI2D model (retrieval + multiple-choice VQA).")
    parser.add_argument("--manifest", type=Path, default=Path("ai2d") / "prepared_v2" / "manifest_hybrid.jsonl")
    parser.add_argument("--split-json", type=Path, default=Path("ai2d") / "prepared_v2" / "split_hybrid.json")
    parser.add_argument("--output-dir", type=Path, default=Path("runs") / "ai2d_hybrid")

    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--batch-size", type=int, default=4)
    parser.add_argument("--eval-batch-size", type=int, default=8)
    parser.add_argument("--num-workers", type=int, default=0)
    parser.add_argument("--lr", type=float, default=2e-4)
    parser.add_argument("--grad-accum-steps", type=int, default=1)
    parser.add_argument("--epochs-stage1", type=int, default=25)
    parser.add_argument("--epochs-stage2", type=int, default=20)
    parser.add_argument("--early-stopping-patience", type=int, default=6)

    parser.add_argument("--hidden-dim", type=int, default=256)
    parser.add_argument("--out-dim", type=int, default=256)
    parser.add_argument("--disable-attn-pool", action="store_true")

    parser.add_argument("--temperature", type=float, default=0.07)
    parser.add_argument("--lambda-ret", type=float, default=0.6)
    parser.add_argument("--lambda-vqa", type=float, default=0.4)
    parser.add_argument("--use-caption-context", action="store_true")

    parser.add_argument("--extract-min-area", type=int, default=300)
    parser.add_argument("--extract-max-nodes", type=int, default=80)
    parser.add_argument("--extract-min-text-conf", type=float, default=35.0)
    parser.add_argument("--extract-knn-k", type=int, default=4)
    parser.add_argument("--ocr-level", choices=["line", "word"], default="line")
    parser.add_argument("--disable-shape-nodes", action="store_true")

    parser.add_argument("--cache-dir", type=Path, default=Path("ai2d") / "_cache_graph_hybrid")
    parser.add_argument("--disable-cache", action="store_true")

    parser.add_argument("--max-train-samples", type=int, default=None)
    parser.add_argument("--max-val-samples", type=int, default=None)
    parser.add_argument("--max-test-samples", type=int, default=None)
    parser.add_argument("--disable-amp", action="store_true")
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    set_seed(args.seed)
    device = torch.device(args.device)

    samples = load_manifest_hybrid(args.manifest)
    split_payload = load_split_payload(args.split_json)

    train_samples = select_samples_for_split(samples, "train", split_payload)
    val_samples = select_samples_for_split(samples, "val", split_payload)
    test_samples = select_samples_for_split(samples, "test", split_payload)

    train_samples = maybe_limit_samples(train_samples, args.max_train_samples, seed=args.seed)
    val_samples = maybe_limit_samples(val_samples, args.max_val_samples, seed=args.seed + 1)
    test_samples = maybe_limit_samples(test_samples, args.max_test_samples, seed=args.seed + 2)

    if not train_samples or not val_samples or not test_samples:
        raise SystemExit(
            "Empty split detected. Check manifest/split json or disable restrictive --max-*-samples values."
        )

    print(f"[INFO] train={len(train_samples)} val={len(val_samples)} test={len(test_samples)}")
    collate_fn = make_hybrid_collate_fn(use_caption_context=args.use_caption_context)

    train_loader = DataLoader(
        Ai2dHybridDataset(train_samples),
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.num_workers,
        collate_fn=collate_fn,
    )
    eval_loader_kwargs = {
        "batch_size": args.eval_batch_size,
        "shuffle": False,
        "num_workers": args.num_workers,
        "collate_fn": collate_fn,
    }
    val_loader = DataLoader(Ai2dHybridDataset(val_samples), **eval_loader_kwargs)
    test_loader = DataLoader(Ai2dHybridDataset(test_samples), **eval_loader_kwargs)

    featurizer = NodeFeaturizerV2(device=str(device))
    cache_signature = build_cache_signature(args, featurizer)
    cache = FeatureCacheV2(
        cache_dir=args.cache_dir,
        signature=cache_signature,
        enabled=not args.disable_cache,
    )

    args.in_dim = featurizer.vision_dim + featurizer.text_dim + 12 + 1 + 1
    gnn = GraphEncoderV2(
        in_dim=args.in_dim,
        hidden_dim=args.hidden_dim,
        out_dim=args.out_dim,
        use_attn_pool=not args.disable_attn_pool,
    ).to(device)
    text_proj = nn.Sequential(
        nn.Linear(featurizer.text_dim, args.hidden_dim),
        nn.GELU(),
        nn.Linear(args.hidden_dim, args.out_dim),
    ).to(device)

    optimizer = torch.optim.AdamW(list(gnn.parameters()) + list(text_proj.parameters()), lr=args.lr)
    use_amp = (device.type == "cuda") and (not args.disable_amp)
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    history: list[dict[str, Any]] = []
    best_composite = float("-inf")
    best_epoch = -1
    no_improve = 0

    checkpoint_path = args.output_dir / "checkpoint_best.pt"
    metrics_path = args.output_dir / "metrics.json"
    args.output_dir.mkdir(parents=True, exist_ok=True)

    stage_plan = [("stage1", args.epochs_stage1), ("stage2", args.epochs_stage2)]
    global_epoch = 0
    stop_training = False

    for stage_name, stage_epochs in stage_plan:
        if stage_epochs <= 0:
            continue
        for local_epoch in range(1, stage_epochs + 1):
            global_epoch += 1
            train_stats = train_epoch(
                loader=train_loader,
                gnn=gnn,
                text_proj=text_proj,
                featurizer=featurizer,
                cache=cache,
                optimizer=optimizer,
                scaler=scaler,
                args=args,
                device=device,
                stage=stage_name,
            )

            val_result = evaluate_split(
                loader=val_loader,
                gnn=gnn,
                text_proj=text_proj,
                featurizer=featurizer,
                cache=cache,
                args=args,
                device=device,
                split_name="val",
            )
            val_metrics = val_result["metrics"]
            entry = {
                "epoch": global_epoch,
                "stage": stage_name,
                "train": train_stats,
                "val": val_metrics,
            }
            history.append(entry)
            print(
                f"[E{global_epoch} {stage_name}] loss={train_stats['loss']:.4f} "
                f"ret={train_stats['loss_ret']:.4f} vqa={train_stats['loss_vqa']:.4f} | "
                f"val MeanR@10={val_metrics['mean']['10']:.4f} "
                f"val VQA={val_metrics['vqa_acc']:.4f} "
                f"val Composite={val_metrics['composite']:.4f}"
            )

            if val_metrics["composite"] > best_composite:
                best_composite = float(val_metrics["composite"])
                best_epoch = global_epoch
                no_improve = 0
                save_checkpoint(
                    path=checkpoint_path,
                    gnn=gnn,
                    text_proj=text_proj,
                    optimizer=optimizer,
                    featurizer=featurizer,
                    cache_signature=cache_signature,
                    args=args,
                    epoch_idx=global_epoch,
                    stage=stage_name,
                    val_metrics=val_metrics,
                )
                print(f"[INFO] New best checkpoint at epoch {global_epoch} (composite={best_composite:.4f})")
            else:
                no_improve += 1
                if no_improve >= args.early_stopping_patience:
                    print(
                        f"[INFO] Early stopping triggered: "
                        f"no improvement for {no_improve} eval steps."
                    )
                    stop_training = True
                    break
        if stop_training:
            break

    if not checkpoint_path.exists():
        raise RuntimeError("Best checkpoint was not saved.")

    checkpoint = torch.load(checkpoint_path, map_location=device)
    gnn.load_state_dict(checkpoint["gnn_state_dict"])
    text_proj.load_state_dict(checkpoint["text_proj_state_dict"])

    test_result = evaluate_split(
        loader=test_loader,
        gnn=gnn,
        text_proj=text_proj,
        featurizer=featurizer,
        cache=cache,
        args=args,
        device=device,
        split_name="test",
    )

    write_jsonl(test_result["vqa_predictions"], args.output_dir / "test_vqa_predictions.jsonl")
    write_jsonl(test_result["retrieval_predictions"], args.output_dir / "test_retrieval_predictions.jsonl")

    metrics_payload = {
        "best_epoch": best_epoch,
        "best_composite": best_composite,
        "train_history": history,
        "test": test_result["metrics"],
        "config": {
            "manifest": str(args.manifest),
            "split_json": str(args.split_json),
            "use_caption_context": bool(args.use_caption_context),
            "lambda_ret": float(args.lambda_ret),
            "lambda_vqa": float(args.lambda_vqa),
            "epochs_stage1": int(args.epochs_stage1),
            "epochs_stage2": int(args.epochs_stage2),
            "batch_size": int(args.batch_size),
            "eval_batch_size": int(args.eval_batch_size),
            "temperature": float(args.temperature),
        },
    }
    metrics_path.write_text(json.dumps(metrics_payload, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"[INFO] Best epoch: {best_epoch}")
    print(f"[INFO] Best composite: {best_composite:.4f}")
    print(f"[INFO] Test composite: {test_result['metrics']['composite']:.4f}")
    print(f"[INFO] Artifacts saved under: {args.output_dir}")




## Eval Script Code

In [ ]:
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path
from typing import Any

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch_geometric.data import Batch
from tqdm.auto import tqdm

ROOT = Path.cwd()
if not (ROOT / "runs").exists() and (ROOT.parent / "runs").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from vqa_retrieval.ai2d_hybrid import (
    Ai2dHybridDataset,
    compute_option_logits,
    load_manifest_hybrid,
    load_split_payload,
    make_hybrid_collate_fn,
    retrieval_metrics_from_embeddings,
    select_samples_for_split,
)
from vqa_retrieval.graph_builder_v2 import FeatureCacheV2, GraphEncoderV2, NodeFeaturizerV2


def write_jsonl(rows: list[dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")


def build_graph_batch(batch, cache, featurizer, cfg, device):
    graphs = [
        cache.get_graph(
            image_path=image_path,
            featurizer=featurizer,
            ocr_path=ocr_path,
            ocr_level=cfg["ocr_level"],
            min_area=cfg["extract_min_area"],
            max_shape_nodes=cfg["extract_max_nodes"],
            min_text_conf=cfg["extract_min_text_conf"],
            knn_k=cfg["extract_knn_k"],
            include_shapes=cfg["include_shapes"],
        )
        for image_path, ocr_path in zip(batch["image_paths"], batch["ocr_paths"])
    ]
    return Batch.from_data_list(graphs).to(device)


def evaluate(
    loader: DataLoader,
    gnn: GraphEncoderV2,
    text_proj: nn.Module,
    featurizer: 'NodeFeaturizerV2',
    cache: FeatureCacheV2,
    cfg: dict[str, Any],
    device: torch.device,
    split_name: str,
):
    gnn.eval()
    text_proj.eval()
    all_img_emb: list[torch.Tensor] = []
    all_q_emb: list[torch.Tensor] = []
    ordered_sample_ids: list[str] = []
    ordered_image_ids: list[str] = []
    vqa_predictions: list[dict[str, Any]] = []

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"[eval:{split_name}]", leave=False):
            batch_graph = build_graph_batch(batch, cache, featurizer, cfg, device)
            q_emb = cache.get_text_batch(batch["question_texts"], featurizer.text_enc, normalize=False).to(device)

            z_img = F.normalize(gnn(batch_graph), dim=1)
            z_q = F.normalize(text_proj(q_emb), dim=1)
            logits = compute_option_logits(
                z_img=z_img,
                option_texts=batch["option_texts"],
                option_mask=batch["option_mask"],
                text_encoder=featurizer.text_enc,
                text_proj=text_proj,
                cache=cache,
                temperature=cfg["temperature"],
            )
            preds = logits.argmax(dim=1)
            targets = batch["correct_indices"].to(device)

            all_img_emb.append(z_img.detach().cpu())
            all_q_emb.append(z_q.detach().cpu())
            ordered_sample_ids.extend(batch["sample_ids"])
            ordered_image_ids.extend(batch["image_ids"])

            for i, sample_id in enumerate(batch["sample_ids"]):
                options = batch["options"][i]
                pred_idx = int(preds[i].item())
                gold_idx = int(targets[i].item())
                pred_text = options[pred_idx] if 0 <= pred_idx < len(options) else ""
                gold_text = options[gold_idx] if 0 <= gold_idx < len(options) else ""
                vqa_predictions.append(
                    {
                        "sample_id": sample_id,
                        "image_id": batch["image_ids"][i],
                        "question": batch["questions"][i],
                        "pred_option_idx": pred_idx,
                        "pred_option_text": pred_text,
                        "gold_option_idx": gold_idx,
                        "gold_option_text": gold_text,
                        "is_correct": bool(pred_idx == gold_idx),
                    }
                )

    z_img = torch.cat(all_img_emb, dim=0)
    z_q = torch.cat(all_q_emb, dim=0)
    retrieval = retrieval_metrics_from_embeddings(
        z_img=z_img,
        z_txt=z_q,
        ks=(1, 5, 10),
        image_ids=ordered_image_ids,
    )
    sim = retrieval["sim"]
    vqa_acc = float(sum(1 for row in vqa_predictions if row["is_correct"]) / max(1, len(vqa_predictions)))
    composite = 0.5 * float(retrieval["mean"][10]) + 0.5 * vqa_acc

    top_k = min(10, sim.size(1))
    top_idx = sim.topk(k=top_k, dim=1).indices
    retrieval_predictions: list[dict[str, Any]] = []
    for row_idx in range(sim.size(0)):
        ranks = [ordered_image_ids[int(col_idx)] for col_idx in top_idx[row_idx].tolist()]
        own_image_id = ordered_image_ids[row_idx]
        retrieval_predictions.append(
            {
                "sample_id": ordered_sample_ids[row_idx],
                "image_id": own_image_id,
                "top10_image_ids": ranks,
                "hit@10": bool(own_image_id in ranks),
            }
        )

    metrics = {
        "split": split_name,
        "i2t": {str(k): float(v) for k, v in retrieval["i2t"].items()},
        "t2i": {str(k): float(v) for k, v in retrieval["t2i"].items()},
        "mean": {str(k): float(v) for k, v in retrieval["mean"].items()},
        "vqa_acc": float(vqa_acc),
        "composite": float(composite),
    }
    return metrics, vqa_predictions, retrieval_predictions


def export_anls_submission(vqa_predictions: list[dict[str, Any]], output_path: Path) -> None:
    rows = []
    for item in vqa_predictions:
        sample_id = str(item["sample_id"])
        _, _, qid = sample_id.partition(":")
        rows.append({"questionId": qid, "answer": item["pred_option_text"]})
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Evaluate trained AI2D hybrid model.")
    parser.add_argument("--manifest", type=Path, default=Path("ai2d") / "prepared_v2" / "manifest_hybrid.jsonl")
    parser.add_argument("--split-json", type=Path, default=Path("ai2d") / "prepared_v2" / "split_hybrid.json")
    parser.add_argument("--checkpoint", type=Path, required=True)
    parser.add_argument("--split", choices=["val", "test"], default="test")
    parser.add_argument("--batch-size", type=int, default=8)
    parser.add_argument("--num-workers", type=int, default=0)
    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    parser.add_argument("--cache-dir", type=Path, default=Path("ai2d") / "_cache_graph_hybrid")
    parser.add_argument("--disable-cache", action="store_true")
    parser.add_argument("--use-caption-context", action="store_true")
    parser.add_argument("--compare-caption-context", action="store_true")
    parser.add_argument("--output-dir", type=Path, default=Path("runs") / "ai2d_hybrid_eval")
    parser.add_argument("--export-anls-submission", type=Path, default=None)
    return parser.parse_args()


def run_eval_once(args, checkpoint, use_caption_context: bool, tag: str):
    model_cfg = checkpoint["model_config"]
    device = torch.device(args.device)

    samples = load_manifest_hybrid(args.manifest)
    split_payload = load_split_payload(args.split_json)
    split_samples = select_samples_for_split(samples, args.split, split_payload)
    if not split_samples:
        raise SystemExit(f"No samples found for split={args.split}")

    collate_fn = make_hybrid_collate_fn(use_caption_context=use_caption_context)
    loader = DataLoader(
        Ai2dHybridDataset(split_samples),
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.num_workers,
        collate_fn=collate_fn,
    )

    featurizer = NodeFeaturizerV2(
        device=str(device),
        vision_model_name=model_cfg["vision_model_name"],
        text_model_name=model_cfg["text_model_name"],
    )
    cache_signature = checkpoint.get("cache_signature")
    if not cache_signature:
        cache_signature = FeatureCacheV2.make_signature(
            vision_model_name=featurizer.vision_model_name,
            text_model_name=featurizer.text_model_name,
            ocr_level=model_cfg["ocr_level"],
            min_area=model_cfg["extract_min_area"],
            max_shape_nodes=model_cfg["extract_max_nodes"],
            min_text_conf=model_cfg["extract_min_text_conf"],
            knn_k=model_cfg["extract_knn_k"],
            include_shapes=model_cfg["include_shapes"],
        )
    cache = FeatureCacheV2(
        cache_dir=args.cache_dir,
        signature=cache_signature,
        enabled=not args.disable_cache,
    )

    gnn = GraphEncoderV2(
        in_dim=model_cfg["in_dim"],
        hidden_dim=model_cfg["hidden_dim"],
        out_dim=model_cfg["out_dim"],
        use_attn_pool=model_cfg["use_attn_pool"],
    ).to(device)
    text_proj = nn.Sequential(
        nn.Linear(featurizer.text_dim, model_cfg["hidden_dim"]),
        nn.GELU(),
        nn.Linear(model_cfg["hidden_dim"], model_cfg["out_dim"]),
    ).to(device)
    gnn.load_state_dict(checkpoint["gnn_state_dict"])
    text_proj.load_state_dict(checkpoint["text_proj_state_dict"])

    eval_cfg = {
        "ocr_level": model_cfg["ocr_level"],
        "extract_min_area": model_cfg["extract_min_area"],
        "extract_max_nodes": model_cfg["extract_max_nodes"],
        "extract_min_text_conf": model_cfg["extract_min_text_conf"],
        "extract_knn_k": model_cfg["extract_knn_k"],
        "include_shapes": model_cfg["include_shapes"],
        "temperature": model_cfg["temperature"],
    }
    metrics, vqa_predictions, retrieval_predictions = evaluate(
        loader=loader,
        gnn=gnn,
        text_proj=text_proj,
        featurizer=featurizer,
        cache=cache,
        cfg=eval_cfg,
        device=device,
        split_name=args.split,
    )

    args.output_dir.mkdir(parents=True, exist_ok=True)
    write_jsonl(vqa_predictions, args.output_dir / f"{args.split}_{tag}_vqa_predictions.jsonl")
    write_jsonl(retrieval_predictions, args.output_dir / f"{args.split}_{tag}_retrieval_predictions.jsonl")
    (args.output_dir / f"{args.split}_{tag}_metrics.json").write_text(
        json.dumps(metrics, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    if args.export_anls_submission is not None:
        export_anls_submission(vqa_predictions, args.export_anls_submission)
    print(
        f"[{tag}] MeanR@10={metrics['mean']['10']:.4f} "
        f"VQA={metrics['vqa_acc']:.4f} Composite={metrics['composite']:.4f}"
    )
    return metrics


def main() -> None:
    args = parse_args()
    checkpoint = torch.load(args.checkpoint, map_location=args.device)

    if args.compare_caption_context:
        run_eval_once(args, checkpoint, use_caption_context=False, tag="q_only")
        run_eval_once(args, checkpoint, use_caption_context=True, tag="q_plus_caption")
    else:
        use_context = bool(args.use_caption_context or checkpoint["model_config"].get("use_caption_context", False))
        tag = "q_plus_caption" if use_context else "q_only"
        run_eval_once(args, checkpoint, use_caption_context=use_context, tag=tag)




## Baseline Script Code

In [ ]:
from __future__ import annotations

import argparse
import json
import random
import re
import sys
from collections import Counter
from pathlib import Path
from typing import Any, Iterable, Sequence

ROOT = Path.cwd()
if not (ROOT / "runs").exists() and (ROOT.parent / "runs").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from vqa_retrieval.ai2d_hybrid import (
    Ai2dHybridSample,
    load_manifest_hybrid,
    load_split_payload,
    select_samples_for_split,
)

TOKEN_RE = re.compile(r"[a-zA-Z0-9]+")


def tokens(text: str) -> set[str]:
    return {tok.lower() for tok in TOKEN_RE.findall(str(text))}


def load_ocr_text(path: str | None) -> str:
    if not path or not Path(path).exists():
        return ""
    payload = json.loads(Path(path).read_text(encoding="utf-8"))
    lines = payload.get("lines", [])
    return " ".join(str(item.get("text", "")) for item in lines)


def load_caption_text(sample: Ai2dHybridSample) -> str:
    return str(sample.short_description or "")


def accuracy(samples: Sequence[Ai2dHybridSample], pred_indices: Sequence[int]) -> float:
    if not samples:
        return 0.0
    hits = sum(int(pred == sample.correct_option_idx) for sample, pred in zip(samples, pred_indices))
    return hits / len(samples)


def random_predictions(samples: Sequence[Ai2dHybridSample], seed: int) -> list[int]:
    rng = random.Random(seed)
    return [rng.randrange(max(1, len(sample.options))) for sample in samples]


def majority_index_predictions(
    train_samples: Sequence[Ai2dHybridSample],
    eval_samples: Sequence[Ai2dHybridSample],
) -> list[int]:
    counts = Counter(sample.correct_option_idx for sample in train_samples)
    default_idx = counts.most_common(1)[0][0] if counts else 0
    preds: list[int] = []
    for sample in eval_samples:
        preds.append(min(default_idx, max(0, len(sample.options) - 1)))
    return preds


def ocr_overlap_predictions(samples: Sequence[Ai2dHybridSample]) -> list[int]:
    preds: list[int] = []
    ocr_cache: dict[str, set[str]] = {}
    for sample in samples:
        if sample.image_id not in ocr_cache:
            ocr_cache[sample.image_id] = tokens(load_ocr_text(sample.ocr_v2_path))
        doc_tokens = ocr_cache[sample.image_id]
        best_idx = 0
        best_score = -1.0
        for idx, option in enumerate(sample.options):
            option_tokens = tokens(option)
            overlap = len(option_tokens & doc_tokens)
            score = overlap / max(1, len(option_tokens))
            if score > best_score:
                best_idx = idx
                best_score = score
        preds.append(best_idx)
    return preds


def image_doc_tokens(samples: Iterable[Ai2dHybridSample]) -> dict[str, set[str]]:
    docs: dict[str, set[str]] = {}
    by_image: dict[str, Ai2dHybridSample] = {}
    for sample in samples:
        by_image.setdefault(sample.image_id, sample)
    for image_id, sample in by_image.items():
        text = " ".join(
            [
                load_caption_text(sample),
                load_ocr_text(sample.ocr_v2_path),
            ]
        )
        docs[image_id] = tokens(text)
    return docs


def lexical_retrieval_metrics(
    samples: Sequence[Ai2dHybridSample],
    docs: dict[str, set[str]],
    ks: Sequence[int] = (1, 5, 10),
    query_mode: str = "q_only",
) -> dict[str, float]:
    hits = {int(k): 0 for k in ks}
    image_ids = sorted(docs)
    for sample in samples:
        if query_mode == "q_plus_correct":
            query_text = f"{sample.question} {sample.correct_option_text}"
        elif query_mode == "q_only":
            query_text = sample.question
        else:
            raise ValueError(f"Unsupported query_mode={query_mode}")
        query_tokens = tokens(query_text)
        scored = []
        for image_id in image_ids:
            doc_tokens = docs[image_id]
            overlap = len(query_tokens & doc_tokens)
            denom = max(1, len(query_tokens | doc_tokens))
            scored.append((overlap / denom, image_id))
        scored.sort(key=lambda item: (-item[0], item[1]))
        ranked = [image_id for _, image_id in scored]
        for k in ks:
            if sample.image_id in ranked[: int(k)]:
                hits[int(k)] += 1
    return {str(k): hits[int(k)] / max(1, len(samples)) for k in ks}


def split_answer_distribution(samples: Sequence[Ai2dHybridSample]) -> dict[str, Any]:
    option_counts = Counter(len(sample.options) for sample in samples)
    correct_idx_counts = Counter(sample.correct_option_idx for sample in samples)
    questions_per_image = Counter(sample.image_id for sample in samples)
    return {
        "num_samples": len(samples),
        "num_images": len(questions_per_image),
        "option_counts": {str(k): int(v) for k, v in sorted(option_counts.items())},
        "correct_index_counts": {str(k): int(v) for k, v in sorted(correct_idx_counts.items())},
        "questions_per_image": {
            "min": min(questions_per_image.values()) if questions_per_image else 0,
            "max": max(questions_per_image.values()) if questions_per_image else 0,
            "mean": sum(questions_per_image.values()) / max(1, len(questions_per_image)),
        },
    }


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Evaluate cheap AI2D VQA/retrieval baselines.")
    parser.add_argument("--manifest", type=Path, default=Path("ai2d") / "prepared_v2" / "manifest_hybrid.jsonl")
    parser.add_argument("--split-json", type=Path, default=Path("ai2d") / "prepared_v2" / "split_hybrid.json")
    parser.add_argument("--split", choices=["val", "test"], default="test")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--retrieval-query-mode", choices=["q_only", "q_plus_correct"], default="q_only")
    parser.add_argument("--output-dir", type=Path, default=Path("runs") / "ai2d_baselines")
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    samples = load_manifest_hybrid(args.manifest)
    split_payload = load_split_payload(args.split_json)
    train_samples = select_samples_for_split(samples, "train", split_payload)
    eval_samples = select_samples_for_split(samples, args.split, split_payload)
    if not eval_samples:
        raise SystemExit(f"No samples found for split={args.split}")

    vqa = {
        "random": accuracy(eval_samples, random_predictions(eval_samples, seed=args.seed)),
        "majority_correct_index": accuracy(eval_samples, majority_index_predictions(train_samples, eval_samples)),
        "ocr_option_overlap": accuracy(eval_samples, ocr_overlap_predictions(eval_samples)),
    }
    retrieval = {
        "lexical_ocr_caption_t2i": lexical_retrieval_metrics(
            eval_samples,
            docs=image_doc_tokens(eval_samples),
            ks=(1, 5, 10),
            query_mode=args.retrieval_query_mode,
        )
    }
    metrics = {
        "split": args.split,
        "seed": args.seed,
        "retrieval_query_mode": args.retrieval_query_mode,
        "distribution": split_answer_distribution(eval_samples),
        "vqa_accuracy": vqa,
        "retrieval": retrieval,
    }

    args.output_dir.mkdir(parents=True, exist_ok=True)
    out_path = args.output_dir / f"{args.split}_metrics.json"
    out_path.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"[INFO] split={args.split} samples={len(eval_samples)}")
    for name, value in vqa.items():
        print(f"[VQA] {name}: {value:.4f}")
    for name, values in retrieval.items():
        print(f"[RET] {name}: R@1={values['1']:.4f} R@5={values['5']:.4f} R@10={values['10']:.4f}")
    print(f"[INFO] metrics={out_path}")




## Monitor Existing Full v4 Run

In [ ]:
if LOG_PATH.exists():
    txt = LOG_PATH.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(txt[-80:]))
else:
    print("No train log yet.")

if METRICS_PATH.exists():
    metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
    print(json.dumps(metrics.get("test", {}), indent=2))
else:
    print("No final metrics yet.")


## Run Full v4 Training From This Notebook

Uncomment the final line to start training. Do not run this while another v4 training process is active.

In [ ]:
cmd = [
    str(ROOT / ".venv" / "Scripts" / "python.exe"),
    str(ROOT / "scripts" / "train_ai2d_hybrid.py"),
    "--output-dir", str(RUN_DIR),
    "--epochs-stage1", "8",
    "--epochs-stage2", "25",
    "--batch-size", "4",
    "--eval-batch-size", "8",
    "--use-caption-context",
]
print(" ".join(cmd))
